# Colab track — no Docker, no GPU, no live OLMo

This is the **demo** copy of `T1-07_data_classification.ipynb`. Replies that would have come from OLMo/Phi are **scripted** (`[OLMo demo]` / `[Phi-4-mini demo]`). Days 2–3 already work without a model.

The live-hosting course is `tier1/` + Docker (or [03_own_environment.md](https://github.com/justkrismanohar/fine-tuning-build/blob/main/03_own_environment.md)).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justkrismanohar/fine-tuning-build/blob/main/colab/day2/T1-07_data_classification.ipynb)

**Runtime → CPU.** Then **Runtime → Run all**. First code cell clones the course (or finds it if you already have the folder).


In [1]:
# [RUN] COLAB TRACK — find or clone the course, stay on demo replies
import os
import sys
import subprocess
from pathlib import Path

os.environ["CIVICDESK_COLAB"] = "1"
os.environ["CIVICDESK_MODE"] = "demo"

def _course_root() -> Path:
    markers = Path("tier1") / "day1" / "civicdesk_lab.py"
    for base in [Path("/content/fine-tuning-build"), Path.cwd(), *Path.cwd().resolve().parents]:
        if (base / markers).is_file():
            return base
    dest = Path("/content/fine-tuning-build")
    url = "https://github.com/justkrismanohar/fine-tuning-build.git"
    print("Cloning", url)
    subprocess.check_call(["git", "clone", "--depth", "1", url, str(dest)])
    return dest

ROOT = _course_root()
DAY = ROOT / "tier1" / "day2"
os.chdir(DAY)
if str(DAY) not in sys.path:
    sys.path.insert(0, str(DAY))
if "day2" == "day3":
    d2 = str(ROOT / "tier1" / "day2")
    if d2 not in sys.path:
        sys.path.insert(0, d2)
print("COLAB TRACK")
print(" course:", ROOT)
print(" cwd   :", Path.cwd())
print(" Live OLMo/Phi are OFF. Later chat cells use canned CivicDesk replies.")

# Markdown images in Colab resolve from /content, not cwd
import shutil
content = Path("/content") if Path("/content").is_dir() else Path.cwd()
for folder in ("images", "cards"):
    src = DAY / folder
    dest = content / folder
    if src.is_dir() and dest.resolve() != src.resolve():
        shutil.copytree(src, dest, dirs_exist_ok=True)
        print(f" {folder} →", dest)

# Day 2 helper lives here; Day 3 notebooks that import rag also add ../day2


COLAB TRACK
 course: G:\Other computers\My Laptop\Phd\UWI\19-2026-2027-SEM-1\15-UNDP Consultant\course
 cwd   : G:\Other computers\My Laptop\Phd\UWI\19-2026-2027-SEM-1\15-UNDP Consultant\course\tier1\day2
 Live OLMo/Phi are OFF. Later chat cells use canned CivicDesk replies.


# T1-07 Data classification

**Time:** 09:20–10:30 &nbsp;|&nbsp; **Evidence:** classification score on CI-01…CI-10

**Notebook learning outcomes**

- Say what an **index** is, in CivicDesk words.
- Label a piece of text **INDEX** or **NEVER**.
- Explain a **canary**, and why clearance is not the same as “allowed in the index.”


**How to use this notebook**

- **No-code / code-light:** run every cell in order. Read the teaching note after a cell. Fill the write-up. Do not paste work files.
- **Occasional-code:** run all cells. Change only lines marked `# [CHANGE]`.
- **Deep-code:** optional cells are tagged `# [OPTIONAL]`. Stay with the room on the required path. Do not drive someone else's notebook.

Git is not required. If you are stuck, load the matching file in `completed/` for this notebook only, mark it as loaded from reference, and continue.


## Overnight readers — the new words

Yesterday the model only saw what you typed. Today you will give it a **corpus**: a pile of approved CivicDesk paragraphs. Before you pile them, you classify.

| Word | Plain meaning | CivicDesk example |
|---|---|---|
| **Corpus** | The set of documents you are willing to search | Twelve fictional handbook chunks, not an EDMS export |
| **Chunk** | One retrieval unit (a paragraph or short section), not a 200-page PDF | `HB-VAC-C2` is one chunk |
| **Index** | The searchable structure over chunks (keyword overlap here; embeddings in production) | Only chunks with `index_ok: true` |
| **Metadata** | Labels on a chunk that are not the paragraph itself | `clearance`, `id`, `citation` |
| **Canary** | Planted sensitive text. If a reply quotes it, the exercise **fails** | National ID `999-00-1111`, name Maya Reid |
| **PII** | Information that identifies a person | CVs, burgess forms, health records, live tickets |

**INDEX** means: this text may be stored in the classroom search pile. **NEVER** means: do not copy it in, even “just to try.”

This is **N5** (data ban) and the start of **N4** (access). T1-10 is the access gate. Today is “does it enter the pile at all?”

![May this enter the CivicDesk index?](../../tier1/day2/images/t2_index_or_not.jpg)


## Ministry link

| Your EOI type | What must never enter an index in *this* room |
|---|---|
| **Labour** | Real CVs, scores, names |
| **Health / THA** | Patient detail, LMS submissions, suicide-guideline *cases* (the *rule* N1 may be indexed) |
| **Finance** | Cabinet *bodies*. Class metadata is a later access question |
| **RDLG** | Burgess applications, household names, addresses |
| **TTDF** | Patrol notes, landing sites, tracks (AUX-05) |
| **Trade / Defence assets** | Live GLPI/serial dumps — query the register; do not index this month’s extract |
| **all** | The canary line. If you index it, you fail the drill |

**N14 woven:** someone must be able to say later “this chunk was / was not in the index.” That is why every chunk has an id.


## 1. Look at the handbook pile (including the canary)

`index_ok` is a flag on the chunk. `clearance` is who may *see* it later. A restricted chunk can still be in the index (T1-10). A prohibited canary must not be.


In [2]:
# [RUN]
import civicdesk_rag as rag

print("All chunks in data/handbooks.json\n")
rag.print_chunks(include_prohibited=True)
print()
print("Indexed only (index_ok True):", len(rag.indexed_chunks()), "of", len(rag.load_chunks()))


All chunks in data/handbooks.json

id           idx clearance    title
HB-VAC-C2    True internal     Vacancy Handbook v3 — criterion C2
HB-ICT-PWD   True internal     ICT knowledge article — passwords
HB-ATTACH-04 True internal     Service-request attachments overnight
HB-NOTICE-N1 True internal     Public notice rule N1
HB-OPR-07    True internal     OPR procurement — no award
HB-EDMS-ACC  True restricted   EDMS access classes
HB-ASSET-Q   True internal     Asset questions — register is source of truth
HB-MIN-02    True internal     Minutes extract rule
HB-CLASS-01  True internal     Incoming request types
HB-PII-00    True internal     What never enters the index
HB-CAB-X     True restricted   Cabinet-class minute (contents restricted)
HB-CANARY    False prohibited   Canary chunk — must never be quoted

Indexed only (index_ok True): 11 of 12


### Teaching note

`HB-CANARY` has `index_ok: false` and `clearance: prohibited`. The helper `retrieve()` skips it unless you pass `include_prohibited=True` (a drill, not a product feature).

`HB-CAB-X` is **restricted** but `index_ok: true`. That is deliberate. The index may know a classed minute exists. Displaying the body is T1-10. Finance-shaped: even that is generous — many rooms would keep Cabinet bodies **out** of the index entirely (see CI-05).


## 2. Classification sheet

Read CI-01…CI-10. Decide INDEX or NEVER **before** you open the next cell if you can.


In [3]:
# [RUN]
import civicdesk_rag as rag

rag.print_classify_sheet()


Label each item INDEX or NEVER. Then fill GUESSES in the next cell.

CI-01  [Labour / Health]
    CivicDesk Vacancy Handbook paragraph on diploma criterion C2.

CI-02  [Labour]
    A real applicant CV exported from a ministry e-recruitment folder.

CI-03  [Trade IMSD]
    ICT knowledge article: password reset is a human identity check.

CI-04  [TTDF]
    Last night's patrol notes with landing sites.

CI-05  [Finance]
    Cabinet Minute body on coastal security, full text.

CI-06  [Finance / MOWI / RDLG]
    CivicDesk Circular HB-ATTACH-04 on overnight attachments.

CI-07  [RDLG]
    Burgess application with household names and an address.

CI-08  [Health / THA]
    Public notice rule N1: no personal medical detail.

CI-09  [Defence assets / Trade]
    GLPI extract of live printer serials for this month.

CI-10  [all]
    A planted line: National ID 999-00-1111 Maya Reid.



## 3. Score your labels

Allowed values: `index` or `never`. The cell ships the teaching key so Run All works. Cover it and try first if you want a real score.


In [4]:
# [CHANGE] replace with your labels to test yourself.
GUESSES = {
    "CI-01": "index",
    "CI-02": "never",
    "CI-03": "index",
    "CI-04": "never",
    "CI-05": "never",
    "CI-06": "index",
    "CI-07": "never",
    "CI-08": "index",
    "CI-09": "never",
    "CI-10": "never",
}

import civicdesk_rag as rag

rag.score_classification(GUESSES)
rag.save_log("T1-07_classify", {"guesses": GUESSES})


CI-01  you=index   gold=index   OK  (Labour / Health)
CI-02  you=never   gold=never   OK  (Labour)
CI-03  you=index   gold=index   OK  (Trade IMSD)
CI-04  you=never   gold=never   OK  (TTDF)
CI-05  you=never   gold=never   OK  (Finance)
CI-06  you=index   gold=index   OK  (Finance / MOWI / RDLG)
CI-07  you=never   gold=never   OK  (RDLG)
CI-08  you=index   gold=index   OK  (Health / THA)
CI-09  you=never   gold=never   OK  (Defence assets / Trade)
CI-10  you=never   gold=never   OK  (all)
score 10/10
saved G:\Other computers\My Laptop\Phd\UWI\19-2026-2027-SEM-1\15-UNDP Consultant\course\tier1\day2\logs\T1-07_classify.json


### After you run

A published **rule** can be indexed. A **person’s file** cannot. A **live extract** of a register cannot — you would be copying operational data into a search pile that the model might quote.

**Write-up:** name one thing from *your* overnight canvas that would be NEVER, in CivicDesk words (no real identifiers).


In [5]:
# [OPTIONAL] what retrieve() would do if someone left the canary searchable
import civicdesk_rag as rag

print("Normal retrieve (canary excluded):")
for h in rag.retrieve("Maya Reid national ID", k=5):
    print(" ", h["id"], "score", h["score"])
print()
print("Unsafe retrieve (include_prohibited=True) — classroom drill only:")
for h in rag.retrieve("Maya Reid national ID", k=5, include_prohibited=True):
    print(" ", h["id"], "score", h["score"], "index_ok", h.get("index_ok"))


Normal retrieve (canary excluded):

Unsafe retrieve (include_prohibited=True) — classroom drill only:
  HB-CANARY score 3 index_ok False
